# 08b — Observation-density / exposure-bias audit

Stage 08a increased tiered usable HOME coverage from 27 to 32 without changing the production core.

Stage 08b tests a different hypothesis:

Does one global HOME/OFFICE gate behave differently for users with very different observation exposure?

The audit separates observation exposure from semantic gate outcome.

Exposure includes total stays, active local dates, HOME-night opportunity dates, OFFICE-weekday opportunity dates, observation span, and recurring-location count.

Gate outcome is decomposed into no opportunity, no recurring candidate, min_dates blocked, share blocked, margin blocked, or emitted.

The most important exposure variables are behavior-window opportunity dates, not raw trajectory-file count. A file count can be high while containing little HOME-night or OFFICE-weekday observation.

Production remains HOME = 27 and OFFICE = 16. No adaptive threshold is introduced in this stage.

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

import pandas as pd

REPO_URL='https://github.com/tanh1c/geolife.git'
BRANCH=os.environ.get('GEOLIFE_REPO_BRANCH','main')
REPO_DIR=Path(os.environ.get('GEOLIFE_REPO_DIR','/root/geolife'))
V=Path('/mnt/geolife-data')
ROOT=V/'cache'/'cp2_v2'
C=ROOT/'08b_exposure_bias_audit'
C.mkdir(parents=True,exist_ok=True)

def run(cmd,cwd=None):
    subprocess.run(cmd,cwd=cwd,check=True)

if not (REPO_DIR/'.git').exists():
    run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)])
else:
    run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH])
    run(['git','-C',str(REPO_DIR),'checkout',BRANCH])
    run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{BRANCH}'])

run([sys.executable,'-m','pip','install','-q','-e','.[dev]'],cwd=REPO_DIR)

for p in (str(REPO_DIR),str(REPO_DIR/'src')):
    if p not in sys.path:
        sys.path.insert(0,p)

from geolife.model import HomeOfficeConfig, build_semantic_locations, infer_home_office

spec=importlib.util.spec_from_file_location(
    's08b',
    REPO_DIR/'analysis'/'08b_exposure_bias_audit.py',
)
assert spec is not None and spec.loader is not None
s08b=importlib.util.module_from_spec(spec)
sys.modules[spec.name]=s08b
spec.loader.exec_module(s08b)

print('sha:',subprocess.run(
    ['git','-C',str(REPO_DIR),'rev-parse','--short','HEAD'],
    capture_output=True,text=True,check=True
).stdout.strip())
print(s08b.synthetic_self_check())

## 1. Frozen input and prerequisite gates

08b rebuilds only the frozen semantic-location representation from the existing 5,821-stay cache.

It also reuses Stage 05 assignments for fixed / HoWDe / recurrence comparator coverage, Stage 08a HOME expansion evidence, and Stage 07j OFFICE near-miss identities.

No raw trajectory scan is required.

In [ ]:
def find_stay_cache():
    candidates=[
        V/'cache'/'03a_user_behavior_deep_dive'/'stays_baseline_v1.pkl',
        V/'artifacts'/'03a'/'stays_baseline_v1.pkl',
        REPO_DIR/'artifacts'/'03a'/'stays_baseline_v1.pkl',
    ]
    for p in candidates:
        if p.exists():
            return p
    hits=sorted(V.glob('**/stays_baseline_v1.pkl'))
    if hits:
        return hits[0]
    raise FileNotFoundError('stays_baseline_v1.pkl')

stays=pd.read_pickle(find_stay_cache())
cfg=HomeOfficeConfig()
semantic_stays,locations=build_semantic_locations(stays,config=cfg)
production=infer_home_office(stays,config=cfg)

counts=production['label'].value_counts().to_dict()
assert len(stays)==5821
assert semantic_stays['user_id'].astype(str).nunique()==136
assert len(locations)==2015
assert counts.get('HOME',0)==27
assert counts.get('OFFICE',0)==16

paths={
    'assignments':ROOT/'05_home_office_reliability_validation'/'assignments_private.pkl',
    'home_expansion':ROOT/'08a_home_coverage_expansion'/'home_expansion_evidence_private.pkl',
    'office_near_miss':ROOT/'07j_near_miss_office_audit'/'near_miss_office_audit_private.pkl',
}
missing=[str(p) for p in paths.values() if not p.exists()]
if missing:
    raise FileNotFoundError('Missing Stage-08b prerequisite cache:\n'+'\n'.join(missing))

assignments=pd.read_pickle(paths['assignments'])
home_expansion=pd.read_pickle(paths['home_expansion'])
office_near_miss=pd.read_pickle(paths['office_near_miss'])

gate=pd.DataFrame([{
    'cp1_stays':len(stays),
    'stay_users':semantic_stays['user_id'].astype(str).nunique(),
    'semantic_locations':len(locations),
    'production_HOME':counts.get('HOME',0),
    'production_OFFICE':counts.get('OFFICE',0),
    'HOME_PROBABLE':int(home_expansion['home_expansion_tier'].eq('HOME_PROBABLE').sum()),
    'office_near_miss_users':int(
        office_near_miss.loc[
            office_near_miss['audit_group'].astype(str).ne('baseline'),
            'user_id'
        ].astype(str).nunique()
    ),
}])
print('Stage-08b input gate: PASS')
display(gate)

## 2. Build per-user exposure profiles

For each of the 136 stay users, compute total stays, active local dates, observation span, recurring locations, total dwell, HOME opportunity dates/hours, and OFFICE opportunity dates/hours.

An opportunity date counts only when at least 600 seconds are actually observed in the relevant behavior window, matching the production minimum relevant-date overlap.

Positive-exposure users are stratified into SPARSE / MEDIUM / DENSE by within-label exposure-date percentile. ZERO remains separate.

These regimes are audit strata, not thresholds.

In [ ]:
profiles=s08b.build_user_exposure_profiles(
    semantic_stays,
    locations,
    production,
    config=cfg,
)

assert len(profiles)==136
assert int(profiles['production_HOME'].sum())==27
assert int(profiles['production_OFFICE'].sum())==16

profile_summary=pd.DataFrame([
    {
        'metric':col,
        'min':float(profiles[col].min()),
        'median':float(profiles[col].median()),
        'max':float(profiles[col].max()),
    }
    for col in [
        'total_stays',
        'active_local_dates',
        'observation_span_days',
        'recurring_locations',
        'home_opportunity_dates',
        'office_opportunity_dates',
    ]
])
display(profile_summary)

print('HOME EXPOSURE REGIME COUNTS')
display(
    profiles['home_exposure_regime']
    .value_counts()
    .reindex(s08b.REGIME_ORDER,fill_value=0)
    .rename_axis('home_exposure_regime')
    .reset_index(name='users')
)

print('OFFICE EXPOSURE REGIME COUNTS')
display(
    profiles['office_exposure_regime']
    .value_counts()
    .reindex(s08b.REGIME_ORDER,fill_value=0)
    .rename_axis('office_exposure_regime')
    .reset_index(name='users')
)

## 3. Decompose the global gate

For every user and label, distinguish no_observed_opportunity, no_recurring_candidate, min_dates_blocked, share_blocked, margin_blocked, share_and_margin_blocked, and emitted.

The unconstrained raw top candidate is retained so we can identify an exposure-limited pattern such as observing only two HOME opportunity nights, seeing the same candidate on both nights, and still failing the global min_dates = 3 rule.

That pattern is not proof that the candidate is semantically correct.

In [ ]:
diagnostics=s08b.build_gate_diagnostics(
    semantic_stays,
    locations,
    profiles,
    config=cfg,
)

home_emitted=int(
    diagnostics.loc[
        diagnostics['label'].eq('HOME'),
        'gate_status'
    ].eq('emitted').sum()
)
office_emitted=int(
    diagnostics.loc[
        diagnostics['label'].eq('OFFICE'),
        'gate_status'
    ].eq('emitted').sum()
)
assert home_emitted==27
assert office_emitted==16

print('GATE STATUS — HOME')
display(
    diagnostics.loc[diagnostics['label'].eq('HOME'),'gate_status']
    .value_counts()
    .rename_axis('gate_status')
    .reset_index(name='users')
)

print('GATE STATUS — OFFICE')
display(
    diagnostics.loc[diagnostics['label'].eq('OFFICE'),'gate_status']
    .value_counts()
    .rename_axis('gate_status')
    .reset_index(name='users')
)

## 4. Emission and gate failure by exposure regime

If the global gate is exposure-sensitive, SPARSE users may be disproportionately blocked by min_dates, while DENSE users may pass min_dates but fail share or margin more often. Emission rates may also rise materially with opportunity exposure.

None of those patterns alone proves unfairness or tells us what an adaptive threshold should be.

In [ ]:
regime_summary=s08b.summarize_gate_by_regime(diagnostics)
display(regime_summary)

## 5. Exposure-limited candidates

Count users blocked by the absolute min_dates = 3 rule even though their unconstrained top recurring candidate explains all, or most, of the behavior-window dates that were actually observed.

This is the clearest diagnostic for a possible sparse-user penalty.

In [ ]:
exposure_limited=s08b.summarize_exposure_limited_cases(diagnostics)
display(exposure_limited)

## 6. Continuous exposure associations

Use Spearman rank correlations to avoid assuming a linear relationship.

Important diagnostics are opportunity dates versus emission, total stays versus emission, opportunity dates versus candidate relevant share, and opportunity dates versus share margin.

A positive exposure-to-emission association suggests coverage depends on observation density. A negative exposure-to-share or exposure-to-margin association would be consistent with dilution among denser/mobile users.

These are descriptive associations, not causal estimates.

In [ ]:
associations=s08b.exposure_association_summary(diagnostics)
display(associations)

## 7. Does independent semantic coverage survive in sparse regimes?

The Stage-05 fixed-window method already contains the global min-date gate. HoWDe-style and recurrence comparators use different support logic.

Compare method selection coverage and exact candidate agreement by exposure regime. If sparse users receive comparator candidates while fixed-window coverage collapses, that is additional evidence that the production gate — not complete absence of semantic signal — contributes to low coverage.

In [ ]:
method_coverage=s08b.method_coverage_by_regime(assignments,profiles)
method_agreement=s08b.fixed_comparator_agreement_by_regime(assignments,profiles)

print('METHOD COVERAGE BY EXPOSURE REGIME')
display(method_coverage)

print('FIXED VS ALTERNATE METHOD AGREEMENT BY EXPOSURE REGIME')
display(method_agreement)

## 8. Where do the five HOME_PROBABLE expansion cases sit?

Stage 08a found five HOME_PROBABLE candidates outside the HOME-27 core.

If they concentrate in SPARSE/MEDIUM exposure regimes, that supports the hypothesis that the strict global gate loses otherwise corroborated HOME candidates when observation opportunity is limited.

If they concentrate in DENSE users, then share/margin dilution or other gate mechanics may be more important.

In [ ]:
home_expansion_regime=s08b.home_expansion_by_regime(
    home_expansion,
    profiles,
)
display(home_expansion_regime)

## 9. Where do the nine OFFICE near-miss users sit?

Stage 07j identified nine one-step OFFICE near misses.

Their exposure regimes help distinguish sparse observation from dense/mobile share-margin dilution. This remains descriptive because Stage 07j already showed that most near misses are not robust enough for automatic promotion.

In [ ]:
office_near_miss_regime=s08b.office_near_miss_by_regime(
    office_near_miss,
    profiles,
)
display(office_near_miss_regime)

## 10. Bias-hypothesis snapshot

This compact table intentionally does not output a binary biased/not biased verdict.

It exposes SPARSE and DENSE emission rates, Spearman correlation between opportunity dates and emission, min_dates-blocked users, and how many blocked users have a raw top candidate covering all observed opportunities.

In [ ]:
bias_snapshot=s08b.build_bias_hypothesis_snapshot(
    regime_summary,
    exposure_limited,
    associations,
)
display(bias_snapshot)

## 11. Save Stage-08b outputs

User-level profiles and gate diagnostics remain private on the Modal Volume. Aggregate summaries are safe for documentation.

In [ ]:
# Private
profiles.to_pickle(C/'user_exposure_profiles_private.pkl')
diagnostics.to_pickle(C/'gate_diagnostics_private.pkl')

# Aggregate
gate.to_csv(C/'input_gate.csv',index=False)
profile_summary.to_csv(C/'exposure_profile_summary.csv',index=False)
regime_summary.to_csv(C/'gate_by_exposure_regime.csv',index=False)
exposure_limited.to_csv(C/'exposure_limited_summary.csv',index=False)
associations.to_csv(C/'exposure_association_summary.csv',index=False)
method_coverage.to_csv(C/'method_coverage_by_exposure_regime.csv',index=False)
method_agreement.to_csv(C/'method_agreement_by_exposure_regime.csv',index=False)
home_expansion_regime.to_csv(C/'home_expansion_by_exposure_regime.csv',index=False)
office_near_miss_regime.to_csv(C/'office_near_miss_by_exposure_regime.csv',index=False)
bias_snapshot.to_csv(C/'bias_hypothesis_snapshot.csv',index=False)

print('saved:',C)

## Interpretation boundary

Stage 08b tests whether global thresholds interact with heterogeneous observation exposure.

It does not assume users with fewer trajectories deserve lower thresholds, equate sparse exposure with semantic certainty, change production gates, promote HOME_PROBABLE into HOME_HIGH_CONFIDENCE, promote OFFICE near misses, or claim causal bias from a correlation alone.

If the measured audit shows strong exposure gradients plus corroborated sparse-user candidates, the next stage should design and validate an exposure-aware emission policy against the frozen global baseline.

If no meaningful gradient appears, the low emission count is more likely driven by semantic ambiguity rather than observation-density bias.